In [1]:
import os
import django

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "protwis.settings")
django.setup()

from build.management.commands.PDB_sequence_helper import *

# construct_structure_annotation_override and fetch_pdb_info
from construct.functions import construct_structure_annotation_override, fetch_pdb_info
from django.shortcuts import get_object_or_404
from protein.models import Protein
from structure.models import Structure
from residue.models import Residue
from common.models import WebLink, WebResource

In [2]:
# Get structure from pdb code
def get_structures_by_pdb_code(pdb_code):
    """
    Retrieve Structure instances associated with a given PDB code.
    """
    try:
        web_resource = WebResource.objects.get(slug="pdb")
        weblink = WebLink.objects.get(web_resource=web_resource, index=pdb_code.upper())
        structures = Structure.objects.filter(pdb_code=weblink)
        return structures
    except (WebResource.DoesNotExist, WebLink.DoesNotExist):
        print(f"No structure found for PDB code {pdb_code}.")
        return Structure.objects.none()


# This function is now simplified, as the main logic moves to gather_sequences_and_distances
def get_wild_type_sequence_for_structure(parent_protein, deletions):
    """
    Builds the wild-type sequence string, excluding residues marked for deletion.
    """
    if not parent_protein:
        return ""

    parent_residues = (
        Residue.objects.filter(protein_conformation__protein=parent_protein)
        .exclude(sequence_number__in=deletions)
        .order_by("sequence_number")
    )
    parent_seq = "".join(res.amino_acid for res in parent_residues)
    return parent_seq


def gather_sequences_and_distances(pdb_code):
    """
    1) Find the Structure matching pdb_code and its parent protein.
    2) Fetch annotations (deletions, segments) using fetch_pdb_info.
    3) Build the filtered wild-type sequence (wt_seq).
    4) Build the filtered PDB sequence (pdb_seq) & CA-CA distances after removing fusion proteins.
    5) Return (wt_seq, pdb_seq, distances, structure, preferred_chain).
    """
    # 1. Get the structure and parent protein
    structures = get_structures_by_pdb_code(pdb_code)
    if not structures.exists():
        raise ValueError(f"No Structure found for PDB code {pdb_code}")
    structure = structures.first()
    parent_protein = structure.protein_conformation.protein.parent

    # Get the chain
    preferred_chain = structure.preferred_chain or "A"
    if "," in preferred_chain:
        preferred_chain = preferred_chain.split(",")[0].strip()

    # 2. Fetch annotations
    d = fetch_pdb_info(pdb_code, parent_protein, preferred_chain=preferred_chain)
    entry_name = d["construct_crystal"]["uniprot"]

    # --- This logic is moved here from the old get_wild_type_sequence_for_structure ---
    deletions = []
    if "deletions" in d:
        for del_range in d["deletions"]:
            if (
                del_range["start"] == 146 and structure.pdb_code.index == "4K5Y"
            ):  # Manual fix
                continue
            for i in range(del_range["start"], del_range["end"] + 1):
                deletions.append(i)

    removed = (
        []
    )  # This list will hold PDB residue numbers to remove (e.g., fusion proteins)
    if "xml_segments" in d:
        for seg in d["xml_segments"]:
            if seg[1]:
                # Logic to identify non-receptor segments
                if (
                    seg[1][0] != entry_name
                    and not seg[-1]
                    and seg[1][0] != "Uncharacterized protein"
                    and "receptor" not in seg[1][0]
                ):
                    if seg[0].split("_")[1] == preferred_chain:
                        for i in seg[6]:
                            removed.append(i)

    removed, deletions = construct_structure_annotation_override(
        structure.pdb_code.index, removed, deletions
    )

    if len(deletions) > len(d["wt_seq"]) * 0.9:
        removed, deletions = [], []
    # --- End of moved logic ---

    # 3. Get the wild-type sequence using the deletions list
    wt_seq = get_wild_type_sequence_for_structure(parent_protein, deletions)

    # 4. Get the PDB sequence and distances, REMOVING fusion proteins
    pdb_text = structure.pdb_data.pdb
    if not pdb_text:
        raise ValueError(
            f"No PDB text stored for Structure {structure.id} ({pdb_code})"
        )

    pdb_seq, distances = generate_seq_and_distances_from_pdb_text(
        pdb_text, preferred_chain, residues_to_remove=removed
    )

    return wt_seq, pdb_seq, distances, structure, preferred_chain


def see_results(pdb_code):
    # This function remains the same
    wt_seq, pdb_seq, distances, structure, chain = gather_sequences_and_distances(
        pdb_code
    )

    print(f"WT seq length:  {len(wt_seq)}")
    print(f"PDB seq length: {len(pdb_seq)}")
    print(f"Distance list:  {len(distances)} elements")

    outlier_indexes = distances_stats(distances)
    ref_seq, temp_seq, pdb_map = run_pairwisealigner(pdb_code, wt_seq, pdb_seq)

    print(f"WT  seq: {wt_seq}")
    print(f"Reference seq: {ref_seq}")

    detect_alignment_mistakes_and_reposition(
        pdb_code,
        wt_seq,
        pdb_seq,
        ref_seq,
        temp_seq,
        pdb_map,
        distances,
        outlier_indexes,
        aanumber=3,
    )

def capture_results(pdb_code):
    """
    Runs the full pipeline for a PDB code and returns a dictionary of results.
    """
    try:
        wt_seq, pdb_seq, distances, structure, chain = gather_sequences_and_distances(
            pdb_code
        )

        outlier_indexes = distances_stats(distances)
        ref_seq, temp_seq, pdb_map = run_pairwisealigner(pdb_code, wt_seq, pdb_seq)

        fixed_temp_seq = detect_alignment_mistakes_and_reposition(
            pdb_code,
            wt_seq,
            pdb_seq,
            ref_seq,
            temp_seq,
            pdb_map,
            distances,
            outlier_indexes,
            aanumber=3,
        )

        return {
            "pdb_code": pdb_code,
            "status": "success",
            "wt_seq_len": len(wt_seq),
            "pdb_seq_len": len(pdb_seq),
            "ref_seq": ref_seq,
            "final_temp_seq": fixed_temp_seq,
        }
    except Exception as e:
        print(f"ERROR processing {pdb_code}: {e}")
        return {"pdb_code": pdb_code, "status": "error", "error_message": str(e)}


In [3]:
pdb_codes = ['7XJJ', '3ODU']
for pdb_code in pdb_codes:
    see_results(pdb_code)

WT seq length:  349
PDB seq length: 269
Distance list:  269 elements
Mean of all distances: 3.8712
Standard deviation of all distances: 0.7102
Lower bound for normal values: 1.7404
Upper bound for normal values: 6.0019
Filtered mean (after removing true outliers): 3.8016
Filtered variance: 0.0006
Filtered standard deviation: 0.0241
True outliers: [12.556099919374839, 11.01201507967285, 6.485127371280845]
Outlier at position 27
Outlier at position 101
Outlier at position 134
WT  seq: MELAVGNLSEGNASWPEPPAPEPGPLFGIGVENFVTLVVFGLIFALGVLGNSLVITVLARSKPGKPRSTTNLFILNLSIADLAYLLFCIPFQATVYALPTWVLGAFICKFIHYFFTVSMLVSIFTLAAMSVDRYVAIVHSRRSSSLRVSRNALLGVGCIWALSIAMASPVAYHQGLFHPRASNQTFCWEQWPDPRHKKAYVVCTFVFGYLLPLLLICFCYAKVLNHLHKKLKNMSKKSEASKKKTAQTVLVVVVVFGISWLPHHIIHLWAEFGVFPLTPASFLFRITAHCLAYSNSSVNPIIYAFLSENFRKAYKQVFKCHIRKDSHLSDTKESKSRIDTPPSTNCTHV
Reference seq: MELAVGNLSEGNASWPEPPAPEPGPLFGIGVENFVTLVVFGLIFALGVLGNSLVITVLARSKPGKPRSTTNLFILNLSIADLAYLLFCIPFQATVYALPTWVLGAFICKFIHYFFTVSMLVSIFTLAAMSVDRYVAIVHSRRSSSLR

In [5]:
import json

pdb_codes = ['7XJJ', '7F8V', '8TB7', '8X79', '8FU6', '8FMZ', '8GTI', '7TS0', '7T10', '6WHA', '8UWL', '8IW4', '8HAF', '8ZSJ', '8H0P', '4L6R', '8GTG', '8XQP', '3V2W', '8W8S', '7VVO', '9JR3', '6LN2', '1GZM', '7YFC', '6RZ5', '7EO4', '7B6W', '6KUX', '6NBI', '7S0F', '6KK1', '6K41', '7X8S', '8UXV', '8XWP', '6ZFZ', '8JWY', '8WVV', '8ID4', '7KI0', '6KJV', '8IWE', '7NA7', '6M1H', '7W6P', '8J23', '8YN4', '8HTI', '8XQO', '8FLQ', '8IRU', '8HN8', '9JR2', '8W8Q', '7PP1', '8WPG', '7WUJ', '8KH5', '6X18', '7KH0', '7SRS', '7EJK', '8JRV', '8JD1', '6NBF', '8W77', '8WKY', '6W25', '6NBH', '8IW1', '7VVK', '5VEX', '8ZFJ', '7VVJ', '8IRS', '3V2Y', '5VEW', '7DUQ', '8TR2', '5WIU', '7NA8', '6TPK', '8YW4', '7T8X', '8HOC', '7W7E', '7SIM', '8W8R', '6LPB', '6ZA8', '7UL2', '7RA3', '7C4S', '8FLS', '6PWC', '7KI1', '3SN6', '8HAO', '7FIY', '8TRD', '7RTB', '7SK5', '7T11', '7EWP', '7VVN', '8GGP', '3C9L', '8QW4', '7F8W', '7UL3', '6DO1', '8UXY', '6KK7', '7ZBE', '8SZF', '8TRC', '8WU1', '6Z4Q', '8A6C', '7EWR', '8YW5', '6KUW', '8TZQ', '8FLU', '7BB6', '6WHC', '5T1A', '8G94', '8YW3', '8PJK', '6ZG9', '7SBF', '8IKH', '7MTQ', '8SZI', '4PHU', '5ZKP', '7SIN', '8JCX', '2YCW', '7VQX', '7YMJ', '7FD9', '8JXW', '7EJA', '5WIV', '6U1N', '5UEN', '8T3Q', '4K5Y', '7M3J', '7WBJ', '7Y36', '7EJ8', '7RBT', '8GTM', '4RWD', '8J22', '7X8R', '7WU9', '7EZC', '7VVL', '8DZS', '8JCV', '7W57', '8YN2', '8J24', '7WIH', '8GGB', '8X7A', '8JWZ', '7SIL', '7VIH', '7EPT', '7UL5', '7RAN', '8V6U', '8GGA', '8HA0', '6ZG4', '8U02', '5JQH', '7EJ0', '9AVL', '8FLR', '8QJ2', '7SHF', '7C61', '8WCB', '8GGE', '7JVR', '8J46', '7VVM', '6KUY', '7Y35']

all_results = []
for pdb_code in pdb_codes:
    print(f"Processing {pdb_code}...")
    result = capture_results(pdb_code)
    all_results.append(result)


with open('results_after.json', 'w') as f:
    json.dump(all_results, f, indent=2)

Processing 7XJJ...
Mean of all distances: 3.8712
Standard deviation of all distances: 0.7102
Lower bound for normal values: 1.7404
Upper bound for normal values: 6.0019
Filtered mean (after removing true outliers): 3.8016
Filtered variance: 0.0006
Filtered standard deviation: 0.0241
True outliers: [12.556099919374839, 11.01201507967285, 6.485127371280845]
Outlier at position 27
Outlier at position 101
Outlier at position 134

=== Detecting alignment mistakes for 7XJJ ===
Number of outliers: 3

Outlier at raw PDB index 27 (aligned pos 67)
Seq after the gap (temp_seq): 'STTNLFILNL'
Seq after the gap (pdb_seq):  'STTNLFILNL'
  No suspicious gap found nearby.

Outlier at raw PDB index 101 (aligned pos 147)
Seq after the gap (temp_seq): 'VSRNALLGVG'
Seq after the gap (pdb_seq):  'VSRNALLGVG'
  Possible mispositioned block (length=1): 'R'
Gap found at alignment pos 140 (length=6 dashes)
Alignment context before gap (temp_seq):           VDRYVAIVHS
Corresponding WT context:                   

/Users/nht435/miniconda3/envs/gpcrdb/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/nht435/miniconda3/envs/gpcrdb/lib/python3.9/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/nht435/miniconda3/envs/gpcrdb/lib/python3.9/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/nht435/miniconda3/envs/gpcrdb/lib/python3.9/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/nht435/miniconda3/envs/gpcrdb/lib/python3.9/site-packages/numpy/core/_methods.py:261: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/nht435/miniconda3/envs/gpcrd

Mean of all distances: nan
Standard deviation of all distances: nan
Lower bound for normal values: nan
Upper bound for normal values: nan
Filtered mean (after removing true outliers): nan
Filtered variance: nan
Filtered standard deviation: nan
True outliers: []
ERROR processing 7B6W: sequence has zero length
Processing 6KUX...
Mean of all distances: 3.8283
Standard deviation of all distances: 0.3439
Lower bound for normal values: 2.7966
Upper bound for normal values: 4.8600
Filtered mean (after removing true outliers): 3.8105
Filtered variance: 0.0001
Filtered standard deviation: 0.0090
True outliers: [10.459099585296764]
Outlier at position 145

=== Detecting alignment mistakes for 6KUX ===
Number of outliers: 1

Outlier at raw PDB index 145 (aligned pos 178)
Seq after the gap (temp_seq): 'PAEPRCEIND'
Seq after the gap (pdb_seq):  'PAEPRCEIND'
  No suspicious gap found nearby.

Processing 6NBI...
Mean of all distances: 3.9236
Standard deviation of all distances: 1.0077
Lower bound for

In [6]:
import json
import difflib


def compare_results(before_file, after_file):
    with open(before_file, "r") as f:
        before_data = json.load(f)
    with open(after_file, "r") as f:
        after_data = json.load(f)

    before_dict = {item["pdb_code"]: item for item in before_data}
    after_dict = {item["pdb_code"]: item for item in after_data}

    changed_cases = 0
    print("--- Starting Comparison ---")

    all_codes = set(before_dict.keys()) | set(after_dict.keys())

    for code in sorted(list(all_codes)):
        before = before_dict.get(code)
        after = after_dict.get(code)

        if (
            not before
            or not after
            or before["status"] == "error"
            or after["status"] == "error"
        ):
            continue

        len_changed = before["pdb_seq_len"] != after["pdb_seq_len"]
        ref_seq_changed = before["ref_seq"] != after["ref_seq"]
        temp_seq_changed = before["final_temp_seq"] != after["final_temp_seq"]

        if len_changed or ref_seq_changed or temp_seq_changed:
            changed_cases += 1
            print(f"\n=== CHANGE DETECTED FOR: {code} ===")
            if len_changed:
                print(
                    f"  PDB Seq Length: {before['pdb_seq_len']} -> {after['pdb_seq_len']}"
                )

            if ref_seq_changed:
                print("  Reference (WT) sequence alignment changed.")

                diff = difflib.unified_diff(
                    before["ref_seq"].splitlines(),
                    after["ref_seq"].splitlines(),
                    fromfile="before",
                    tofile="after",
                )
                print("\n".join(diff))

            if temp_seq_changed:
                print("  Template (PDB) sequence alignment changed.")

    print(f"\n--- Comparison Complete ---")
    print(f"Total cases with changes: {changed_cases}")


compare_results("results_before.json", "results_after.json")


--- Starting Comparison ---

=== CHANGE DETECTED FOR: 3SN6 ===
  PDB Seq Length: 443 -> 284
  Reference (WT) sequence alignment changed.
--- before

+++ after

@@ -1 +1 @@

----------------------------------------------------------------------------------------------------------------------------------------------------------------EVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILMKMWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINCYANETCCDFFTNQAYAIASSIVSFYVPLVIMVFVYSRVFQEAKRQLQKIDKSEGRFHVQNLSQVEQDGRTGHGLRRSSKFCLKEHKALKTLGIIMGTFTLCWLPFFIVNIVHVIQDNLIRKEVYILLNWIGYVNSGFNPLIYCRSPDFRIAFQELLCLRRSSLKAYGNGYSSNGNTGEQSG
+EVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILMKMWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINCYANETCCDFFTNQAYAIASSIVSFYVPLVIMVFVYSRVFQEAKRQLQKIDKSEGRFHVQNLSQVEQDGRTGHGLRRSSKFCLKEHKALKTLGIIMGTFTLCWLPFFIVNIVHVIQDNLIRKEVYILLNWIGYVNSGFNPLIYCRSPDFRIAFQELLCLRRSSLKAYGNGYSSNG